# 01_data_construction

Build the person-year master table and the transition-pair table from KHPS 2nd-phase IND files (2019-2024, prefixes a-f).

In [1]:
# 01_data_construction.ipynb
# Integrate KHPS wave IND files into a long person-year master table and a
# transition-pair table used for incidence (new-onset) analysis.

import os
import numpy as np
import pandas as pd
import pyreadstat

ROOT = os.path.abspath("..")
RAW  = os.path.join(ROOT, "data", "raw")
DATA = os.path.join(ROOT, "data")

# Wave prefix mapping: 2019=a ... 2024=f
FILES = [(2019, "a"), (2020, "b"), (2021, "c"),
         (2022, "d"), (2023, "e"), (2024, "f")]

# Variables to pull from each IND file
USECOLS = ["PIDWON", "HHID", "SEX", "BIRTH_Y", "EDU", "ECO1",
           "CD1_HTN", "CD1_DM", "CD1_DYS",
           "HT", "WT", "S1", "S3", "D1", "D4", "D6", "D7", "P1", "P2", "WTMG",
           "I_WGC", "I_WSC", "H_INC_TOT", "REGION1"]

def clean_codes(s):
    # Treat survey missing codes (-9 dont know / refuse, -8, -1) as NaN
    return s.mask(s.isin([-9, -8, -1]))

frames = []
for year, pfx in FILES:
    path = os.path.join(RAW, f"y{year}", f"{pfx}_ind.sas7bdat")
    _, meta = pyreadstat.read_sas7bdat(path, metadataonly=True)
    have = [c for c in USECOLS if c in meta.column_names]
    d, _ = pyreadstat.read_sas7bdat(path, usecols=have)
    for c in USECOLS:
        if c not in d.columns:
            d[c] = np.nan  # variable absent in this wave (e.g. DYS before 2024)

    d["year"] = year
    d["age"]  = year - d["BIRTH_Y"]

    for c in ["S1", "S3", "D1", "D4", "D6", "D7", "P1", "P2", "WTMG", "EDU", "ECO1"]:
        d[c] = clean_codes(d[c])

    # BMI from self-reported height/weight, with plausibility bounds
    ht = d["HT"].where((d["HT"] > 100) & (d["HT"] < 250))
    wt = d["WT"].where((d["WT"] > 20)  & (d["WT"] < 250))
    d["BMI"] = (wt / (ht / 100) ** 2).where(lambda x: (x > 10) & (x < 60))

    # Smoking status: combine lifetime (S1) and current type (S3)
    # current smoker = S3 in {1,2}; non-smoker = S3==3 (former) or S1==3 (never)
    d["smoke_cur"] = np.select(
        [d["S3"].isin([1, 2]), (d["S3"] == 3) | (d["S1"] == 3)], [1, 0], default=np.nan)

    # Drinking (only meaningful where D1 exists, i.e. not 2024)
    d["drink_hi"] = np.where(d["D1"].isin([6, 7, 8]), 1,
                     np.where(d["D1"].isin([1, 2, 3, 4, 5]), 0, np.nan))
    d["binge"] = d["D7"]

    # Physical activity
    d["exer_reg"]  = np.select([d["P1"] == 1, d["P1"] == 2], [1, 0], default=np.nan)
    d["walk_days"] = d["P2"].replace(8, 0)   # 8 = never -> 0 days

    # Chronic disease flags: 1=yes -> 1, 2=no -> 0, else NaN
    for dz in ["CD1_HTN", "CD1_DM", "CD1_DYS"]:
        d[dz] = d[dz].map({1: 1, 2: 0})

    frames.append(d)

panel = pd.concat(frames, ignore_index=True).rename(
    columns={"CD1_HTN": "HTN", "CD1_DM": "DM", "CD1_DYS": "DYS"})

keep = ["PIDWON", "HHID", "year", "age", "SEX", "EDU", "ECO1",
        "HTN", "DM", "DYS", "BMI", "HT", "WT", "smoke_cur",
        "drink_hi", "binge", "exer_reg", "walk_days", "WTMG",
        "I_WGC", "I_WSC", "H_INC_TOT", "REGION1"]
panel = panel[keep]
panel.to_parquet(os.path.join(DATA, "khp_panel_long.parquet"))
print("panel:", panel.shape, "| adults:", (panel.age >= 19).sum(),
      "| unique:", panel.PIDWON.nunique())

panel: (86569, 23) | adults: 68211 | unique: 21651


In [2]:
# Build transition-pair table: adjacent waves, adult baseline.
# Each row is one person-interval carrying t0 features and t1 disease status.

adult = panel[panel.age >= 19]
PAIRS = [(2019, 2020), (2020, 2021), (2021, 2022), (2022, 2023), (2023, 2024)]

trans = []
for t0, t1 in PAIRS:
    a = adult[adult.year == t0]
    b = panel[panel.year == t1][["PIDWON", "HTN", "DM", "age"]]
    m = a.merge(b, on="PIDWON", suffixes=("", "_t1"))
    m["interval"] = f"{t0}-{t1}"
    m["t0"] = t0
    trans.append(m)

T = pd.concat(trans, ignore_index=True)
T.to_parquet(os.path.join(DATA, "khp_transitions.parquet"))

# New-onset (incidence) among those at risk (t0 disease-free)
for dz in ["HTN", "DM"]:
    risk = T[T[dz] == 0]
    inc  = (risk[f"{dz}_t1"] == 1).mean() * 100
    print(f"{dz}: at-risk {len(risk)}, new onset {(risk[f'{dz}_t1']==1).sum()}, "
          f"pooled incidence {inc:.2f}%")

HTN: at-risk 32130, new onset 901, pooled incidence 2.80%
DM: at-risk 41273, new onset 488, pooled incidence 1.18%


In [ ]:
# Persist Table 1 (sample sizes) for the results folder.
import os
TAB = os.path.join(ROOT, "results", "tables")
os.makedirs(TAB, exist_ok=True)
tab1 = pd.DataFrame({
    "Item": ["Person-year (total)", "Adult (19+)", "Unique individuals",
             "Transition pairs", "Step4 3-wave sample"],
    "N": [len(panel), int((panel.age>=19).sum()), panel.PIDWON.nunique(),
          len(T), 21882],
})
tab1.to_csv(os.path.join(TAB, "table1_sample.csv"), index=False)
print(tab1.to_string(index=False))